# Classifier Analysis

Runs Global Logit Impact (with Reactome enrichment), Cluster TF Impact (IG + attention), and per-cluster pathway enrichment (IG + attention).

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
import scanpy as sc

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from model.utils.config import load_yaml_config
from model.utils.constants import get_pv_clusters, get_npv_clusters
from model.models import Classifier, ClassifierConfig
from model.training.checkpointing import load_checkpoint
from model.data.preprocessing import prepare_clusters
from model.analysis import (
    GlobalLogitImpactEngine,
    ClusterTFImpactEngine,
    EnrichmentConfig,
    PathwayEnrichmentRunner,
    run_cluster_enrichment_from_df,
)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE:', DEVICE)

In [ ]:
# ===== Parameters =====
SMOKE_MODE = True            # False -> more complete run
RUN_ENRICHMENT = True        # requires gseapy + network access for Enrichr

N_REPEATS = 2 if SMOKE_MODE else 10
N_PER_CLASS = 80 if SMOKE_MODE else 300
N_CELLS_PER_CLUSTER = 50 if SMOKE_MODE else 200
IG_STEPS = 20 if SMOKE_MODE else 50

OUTDIR = ROOT / 'artifacts' / 'notebooks' / 'classifier'
OUTDIR.mkdir(parents=True, exist_ok=True)

CFG = load_yaml_config(ROOT / 'configs' / 'classifier_binary.yaml')
adata_path = ROOT / CFG['data']['h5ad_path']
ckpt_path = ROOT / CFG['training']['checkpoint_path']
tf_file = ROOT / 'data' / 'Homo_sapiens_TF.html'

print('adata_path:', adata_path)
print('ckpt_path:', ckpt_path)
print('tf_file:', tf_file)

In [ ]:
adata = sc.read_h5ad(adata_path)
adata = prepare_clusters(adata)
print(adata)

model_cfg = dict(CFG['model'])
if model_cfg.get('vocab_size') in [None, 'null']:
    model_cfg['vocab_size'] = int(adata.n_vars) + 1

classifier = Classifier(ClassifierConfig.from_dict(model_cfg)).to(DEVICE)
classifier, ckpt = load_checkpoint(classifier, ckpt_path, device=DEVICE)
classifier.eval()
print('classifier loaded. checkpoint keys:', list(ckpt.keys())[:8])

## 1) Global Logit Impact

In [ ]:
gli_engine = GlobalLogitImpactEngine(
    model=classifier,
    adata=adata,
    device=DEVICE,
    output_dir=str(OUTDIR / 'LogitImpactResults'),
)
repeat_df, summary_df, meta_df = gli_engine.run_repeated_subsampling(
    n_repeats=N_REPEATS,
    n_per_class=N_PER_CLASS,
    replace=False,
    random_state=42,
    save_prefix='global_logit_impact',
    verbose=True,
)

display(summary_df.head(20))
summary_df.to_csv(OUTDIR / 'global_logit_impact_summary.csv', index=False)
print('Saved:', OUTDIR / 'global_logit_impact_summary.csv')

### 1a) Logit Impact Pathway Enrichment

Reactome pathway over-representation on top logit-impact genes (ranked by MeanAbsDelta).

In [ ]:
if RUN_ENRICHMENT:
    logit_runner = PathwayEnrichmentRunner(
        config=EnrichmentConfig(output_dir=str(OUTDIR / 'EnrichmentResults'), verbose=True)
    )
    enrich_logit = logit_runner.run_from_df(
        df=summary_df,
        gene_col='Gene',
        score_col='MeanAbsDelta',
        top_n=80 if SMOKE_MODE else 150,
        min_freq_col='MeanFrequency',
        min_freq=0.05,
        title='Reactome enrichment: global logit-impact genes',
    )
    display(enrich_logit.get('sig_results', pd.DataFrame()).head(20))
    logit_runner.save_results(
        enrich_res=enrich_logit,
        filename='reactome_logit_sig_results.csv',
    )
    print('Saved logit enrichment to:', str(OUTDIR / 'EnrichmentResults'))
else:
    print('RUN_ENRICHMENT=False -> skipped')

## 2) Cluster TF Impact

Integrated gradient and attention attribution per cluster using repeated subsampling.

In [ ]:
cluster_path_dict = {c: 'PV_path' for c in get_pv_clusters()}
cluster_path_dict.update({c: 'NPV_path' for c in get_npv_clusters()})

cti_engine = ClusterTFImpactEngine(
    model=classifier,
    adata=adata,
    device=DEVICE,
    tf_file=str(tf_file),
    output_dir=str(OUTDIR / 'TFresults'),
)
repeat_df_tf, summary_df_tf, cluster_sum_df_tf, meta_df_tf = cti_engine.run_repeated_subsampling(
    cluster_path_dict=cluster_path_dict,
    n_repeats=N_REPEATS,
    n_cells_per_cluster=N_CELLS_PER_CLUSTER,
    replace=False,
    ig_steps=IG_STEPS,
    file_prefix='branch_tf_impact',
    verbose=True,
)

display(cluster_sum_df_tf.head(20))
cluster_sum_df_tf.to_csv(OUTDIR / 'branch_tf_impact_cluster_summary.csv', index=False)
print('Saved:', OUTDIR / 'branch_tf_impact_cluster_summary.csv')

## 3) Pathway Enrichment (Optional)

Reactome pathway over-representation on IG and attention scores from the TF impact analysis.

### 3a) IG-based Cluster Enrichment

Reactome enrichment per cluster on integrated gradient scores (ranked by Abs_Mean_IG_to_PV).

In [ ]:
if RUN_ENRICHMENT:
    runner = PathwayEnrichmentRunner(
        config=EnrichmentConfig(output_dir=str(OUTDIR / 'EnrichmentResults'), verbose=True)
    )
    enr_df, cluster_gene_dict = run_cluster_enrichment_from_df(
        runner=runner,
        df=cluster_sum_df_tf,
        cluster_col='Cluster',
        gene_col='Gene',
        score_col='Abs_Mean_IG_to_PV',
        top_n=80 if SMOKE_MODE else 150,
        min_freq_col='Frequency',
        min_freq=0.05,
        absolute_score=False,
        verbose=True,
    )
    display(enr_df.head(20))
    enr_df.to_csv(OUTDIR / 'cluster_enrichment_results.csv', index=False)
    print('Saved:', OUTDIR / 'cluster_enrichment_results.csv')
else:
    print('RUN_ENRICHMENT=False -> skipped')

### 3b) Attention-based Cluster Enrichment

Reactome enrichment per cluster on attention scores (ranked by Mean_Attn).

In [ ]:
if RUN_ENRICHMENT:
    # Reuse runner from IG enrichment or create new one
    attn_runner = PathwayEnrichmentRunner(
        config=EnrichmentConfig(output_dir=str(OUTDIR / 'EnrichmentResults'), verbose=True)
    )
    enr_attn_df, attn_gene_dict = run_cluster_enrichment_from_df(
        runner=attn_runner,
        df=cluster_sum_df_tf,
        cluster_col='Cluster',
        gene_col='Gene',
        score_col='Mean_Attn',
        top_n=80 if SMOKE_MODE else 150,
        min_freq_col='Frequency',
        min_freq=0.05,
        absolute_score=False,
        verbose=True,
    )
    display(enr_attn_df.head(20))
    enr_attn_df.to_csv(OUTDIR / 'cluster_attention_enrichment_results.csv', index=False)
    print('Saved:', OUTDIR / 'cluster_attention_enrichment_results.csv')
else:
    print('RUN_ENRICHMENT=False -> skipped')